---

### API Struktur

#### 1. Einbau der API, um Requests schicken zu können
#### 2. Wir geben Suchkriterien ein (params), in einer strukturierten Form
#### 3. Response der API in dict liste speichern
#### 4. Response der API in Begriffe normalisieren, die für alle APIs relevant sind, z.B wird hier company_name in company umgewandelt.
---
#### 1. API Aufruf = URL + Response
#### 2. Suchkriterien: params = get_params(search, category, company, limit)
#### 3. Response: raw_jobs = fetch_remotive(params)
#### 4. Normalisierung: jobs = normalize_remotive_list(raw_jobs)

#### Diese Sruktur variiert jeweils vom API Abruf bis zur Response.
#### Die Normalisierung sorgt dafür, das die Daten aus allen APIs einheitlich zusammengefügt werden können.

---

### 1. API Einrichtung

#### Imports

In [1]:
import json
from datetime import datetime
import requests
import pandas as pd

#### API

In [2]:
remotive_URL= "https://remotive.com/api/remote-jobs"
response= requests.get(remotive_URL)
print(response.status_code)


200


### 2. Suchkriterien

#### User Inputs

In [3]:
search =input("Suchbegriff eingeben (leer lassen zum Überspringen): ")
category = input("Kategorie eingeben (z. B. software-dev, leer lassen): ")
company = input("Firmenname eingeben (leer lassen): ")
limit = input("Limit Anzahl Ergebnisse (z. B. 10, leer lassen): ") or 5

#### get_params()

>### Variablen aus User Input als Parameter für die API festlegen
>#### If - Funktion um die Variablen für die API verständlich zu machen
>#### Bsp.: if search = nicht leer -> True, wenn None oder "" ("" = leer) -> False
>#### strip() = entfernt die Leerzeichen am Anfang und Ende
>#### Die Funktion überprüft das die Variablen nicht leer sind, nachdem die Leerzeichen entfernt wurden.
>#### API's mögen keine Leere Variablen, deswegen diese Funktion.

In [4]:
def  get_params(search: str, category : str, company :str, limit : int | None = None )-> dict:
    params = {}
    if search and search.strip():
        params["search"] = search.strip()
    if category and category.strip():
        params["category"] = category.strip()
    if company and company.strip():
        params["company_name"] = company.strip()
    if limit is not None:
        params["limit"] = int(limit)
    return params


In [5]:
def fetch_remotive(params: dict) -> list[dict]:
    # API aufrufen und in das Normalformat umwandeln.
    # params wird dieser als dict angegeben, und wenn mehr as ein dict gibt (ein job= ein dict), wird eine liste von dicts erstellt.
    r = requests.get(remotive_URL, params=params, timeout=30)
    #Anfrage (request) mit api url, unsere userinput params (recherche Angaben), wartelimit (timeout=30)
    r.raise_for_status()
    # test der Verbindung. Wenn 200, funktionniert. Wird nicht angezeigt hier. braucht dafür ein print(r.raise_for_status())
    data = r.json()
    #speichern der Daten in data Variabeln als json Format
    return r.json().get("jobs", [])
    # braucht ein return

### 4. Normalisierung

In [6]:
def normalize_remotive(job: dict) -> dict:
    # Normalisierung der Suchbegriffe für ein dict. Nicht für die Dict liste !
    return {
        "id": f"remotive:{job.get('id')}",
        "source": "remotive",
        "title": job.get("title"),
        "company": job.get("company_name"),
        "location": job.get("candidate_required_location"),
        "url": job.get("url"),
        "posted_at": job.get("publication_date"),
    }

In [7]:
# Wir müssen noch die normalize_remotive einbauen, sodass sie die ganze liste Normalisiert, und nicht nur das erste dict unserer Liste
def normalize_remotive_list(jobs: list[dict]) -> list[dict]:
    return [normalize_remotive(j) for j in jobs]

In [8]:
# Hier bauen wir die Funktionen  in usere Strukturlogik ein

params = get_params(search, category, company, limit)
raw_jobs = fetch_remotive(params)
jobs = normalize_remotive_list(raw_jobs)

In [9]:
# Befehl an der API mit usere Suchkriterien (params)  und lassen uns die url anzeigen zum Controlling
r = requests.get(remotive_URL, params=params, timeout=30)
r.raise_for_status()
print("Aufgerufene URL:", r.url)

Aufgerufene URL: https://remotive.com/api/remote-jobs?limit=5


In [10]:
"""data = r.json()
for j in data.get("jobs", [])[:5]:
    print("-----")
    print("Titel:", j.get("title"))
    print("Unternehmen:", j.get("company_name"))
    print("Ort:", j.get("candidate_required_location"))
    print("Link:", j.get("url"))
"""
# Wir lassen uns die r (response) in ein DF angeben
jobs= normalize_remotive_list(raw_jobs)
pd.DataFrame(jobs)

,id,source,title,company,location,url,posted_at
0,remotive:1987878,remotive,Language teachers,AE Virtual Class S.A,Americas,https://remotive.com/remote-jobs/all-others/la...,2025-09-21T05:30:59
1,remotive:1359476,remotive,Senior DevOps Engineer (AWS),Proxify,CET +/- 3 HOURS,https://remotive.com/remote-jobs/software-dev/...,2025-09-18T07:31:22
2,remotive:1919265,remotive,Senior Independent Software Developer,A.Team,"Americas, Europe, Israel",https://remotive.com/remote-jobs/software-dev/...,2025-09-16T10:15:55
3,remotive:1919266,remotive,Senior Independent AI Engineer / Architect,A.Team,"Americas, Europe, Israel",https://remotive.com/remote-jobs/software-dev/...,2025-09-16T10:15:46
4,remotive:1185979,remotive,Freelance Writer,IAPWE,Worldwide,https://remotive.com/remote-jobs/writing/freel...,2025-09-04T17:00:47
